[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/00_getting_started.ipynb)

# 0 · Getting started

*Brown Dwarf Spectra On-Ramp · about 20 minutes*

**By the end of this notebook you will be able to:**
- run cells in Google Colab and save your own copy in Google Drive
- restart a notebook and get back to where you were
- say what a spectrum file holds, and what F<sub>λ</sub>, F<sub>ν</sub>, and µm mean
- read a spectrum plot on linear and logarithmic axes

> **First time here?** Do *File → Save a copy in Drive* before you change anything (notebook 0 explains why).
> Then run the **Setup** cell below. Run cells in order, top to bottom, with **Shift + Enter**.
> **Moving between notebooks:** use the links inside the notebooks (such as **Next** at the bottom). They always open
> the original notebook, so they work from your copy too — then save a copy of the new notebook before you edit it.

## 1. Working in Colab

Colab runs Python on a Google computer. That computer is **temporary**: when you close the tab
or stay idle for a while, it is wiped. Two consequences:

1. **Save your own copy before editing.** *File → Save a copy in Drive.* A tab opens titled
   "Copy of …" — work in *that* tab. Your edits and answers are saved in your Google Drive
   (folder *Colab Notebooks*). If you skip this, your changes disappear.
2. **Every time you come back, run the Setup cell again.** The notebook text is saved, but
   the variables and downloaded files are not.
3. **Use the links to move between notebooks.** The **Next** link at the bottom of each notebook opens the
   *original* version of the next notebook (from any copy). Save a copy of that one too before you edit it.

| To… | Do this |
|---|---|
| run one cell | click it, press **Shift + Enter** |
| run everything from the top | *Runtime → Run all* |
| start fresh after things get confusing | *Runtime → Restart session*, then Run all |
| see help on a function | run `help(bd.plot_spectrum)` or `bd.plot_spectrum?` |

> If a cell shows ⚠️ and a short message, read the message — it says what to fix.
> ⏳ means some data has not been supplied by your instructor yet; just keep going.

In [ ]:
# ▶ SETUP - run this cell first. It is safe to run more than once.
REPO_URL = "https://github.com/zshakespear/bd-spectra-onramp.git"
import os, subprocess, sys
from pathlib import Path

if "google.colab" in sys.modules:                       # running in Google Colab
    ROOT = Path("/content/bd-spectra-onramp")
    if not (ROOT / "bdtools").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(ROOT), "pull", "--ff-only", "--quiet"], check=False)
else:                                                   # running in Jupyter on your own computer
    ROOT = Path.cwd()
    while not (ROOT / "bdtools").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import matplotlib.pyplot as plt
import bdtools as bd
bd.setup_notebook()

If the last line says `bdtools … ready`, you are set up. `bd` is the small helper library
for this course; `np` is numpy (numbers) and `plt` is matplotlib (plots).

## 2. What a spectrum file holds

A **spectrum** is a list of brightness measurements at different wavelengths. Our
reference spectra come from **SPHEREx**, a NASA space telescope that measures every
part of the sky at 102 wavelengths ("channels") between 0.75 and 5 µm
(1 µm = 1 micron = 10<sup>−6</sup> m; visible light is 0.4–0.7 µm, so all of this is infrared).

In [ ]:
# ✏️ SETTINGS YOU CAN CHANGE
SPECTRAL_TYPE = "L5"     # which reference spectrum to look at

In [ ]:
spec = bd.load_reference(SPECTRAL_TYPE)
print(spec)
spec.meta     # everything else we know about it

Each spectrum has three columns: wavelength, flux, and the flux **uncertainty** (the 1σ error bar).
Let's look at the first few rows as a table:

In [ ]:
import pandas as pd
pd.DataFrame({"wavelength (um)": spec.wave_um, "flux": spec.flux, "uncertainty": spec.err}).head(8)

### Units

* **F<sub>λ</sub>** ("F-lambda"): energy per second, per square metre of telescope, *per micron
  of wavelength*. Unit: W m<sup>−2</sup> µm<sup>−1</sup>. Our files use this.
* **F<sub>ν</sub>** ("F-nu"): the same, but *per hertz of frequency*. Astronomers often quote it
  in **janskys** (1 Jy = 10<sup>−26</sup> W m<sup>−2</sup> Hz<sup>−1</sup>). The SPHEREx papers use Jy.

They describe the same light, but the shapes of the curves differ, because a micron of
wavelength covers more hertz at short wavelengths than at long ones. bdtools converts for you:

In [ ]:
spec_jy = spec.to_fnu_jy()
print(spec_jy)
print("flux at the first channel:", spec.flux[0], "W/m2/um  =", spec_jy.flux[0], "Jy")

## 3. Reading a spectrum plot

The pale **coloured shading** marks wavelengths where different molecules absorb light; the key above the plot
says which colour is which molecule (more on these in notebook 3). The narrow grey strip that hugs the line is the
±1σ uncertainty.

In [ ]:
fig, ax = bd.plot_spectrum(spec, title=f"{spec.name}  (linear flux axis)")

**Predict, then run.** On a *logarithmic* flux axis, equal steps up the axis mean equal
*factors* of brightness (×10, ×100), not equal amounts. Before running the next cell:
will the faint parts of the spectrum (beyond 3 µm) look more or less important?

In [ ]:
fig, ax = bd.plot_spectrum(spec, log=True, title=f"{spec.name}  (log flux axis)")

On the linear plot the long-wavelength part looks almost flat and empty. On the log plot you
can see its shape: dips and bumps that are invisible on a linear axis. **We will use both
axes all the time.** Linear shows where most of the light is; log shows the structure in faint regions.

Same spectrum in F<sub>ν</sub>:

In [ ]:
fig, ax = bd.plot_spectrum(spec, unit="Jy", log=True, title=f"{spec.name} in F_nu (Jy)")

### Exercise 0.1
Using the linear-axis plot, at roughly what wavelength is this object brightest in F<sub>λ</sub>?
Now look at the F<sub>ν</sub> plot. Is the brightest wavelength the same?

<details><summary><b>Answer</b> (try first, then click)</summary>

In F<sub>λ</sub> the L5 spectrum peaks near 1.2–1.3 µm (the "J peak"). In F<sub>ν</sub> the peak
moves to longer wavelength, because F<sub>ν</sub> = F<sub>λ</sub> λ²/c weights long wavelengths up.
"Where is it brightest?" therefore depends on the unit — always say which one you mean.

</details>

### Exercise 0.2
Change `SPECTRAL_TYPE` in the settings cell to `"L0"`, then re-run the cells below it.
How does the L0 spectrum differ from the L5 on the log plot?

<details><summary><b>Answer</b> (try first, then click)</summary>

The L0 is bluer: relatively more light at short wavelengths and a steeper drop toward 5 µm,
because it is hotter. You will see the whole sequence in notebook 4.

</details>

## Summary

- Colab is temporary: save a copy to Drive, and re-run Setup each time you return.
- A spectrum = wavelengths, fluxes, and uncertainties, with units attached.
- F<sub>λ</sub> (W m⁻² µm⁻¹) and F<sub>ν</sub> (Jy) describe the same light but peak in different places.
- Linear axes show where the light is; log axes show structure in faint regions.

**Field guide:** the pages linked above are the short versions to keep open while you work.

## Self-check

Answer these without scrolling up. If one is hard, revisit that section.

1. Why must you save a copy to Drive before editing?
<details><summary><b>Answer</b> (try first, then click)</summary>

Colab's computer is wiped when the session ends; only a copy saved in your Drive keeps your work.

</details>

2. What do you need to do every time you reopen a notebook?
<details><summary><b>Answer</b> (try first, then click)</summary>

Run the Setup cell (and then the cells you need) again.

</details>

3. What is the difference between F<sub>λ</sub> and F<sub>ν</sub>?
<details><summary><b>Answer</b> (try first, then click)</summary>

Flux per unit wavelength versus per unit frequency. Same light, different shape; F<sub>ν</sub> = F<sub>λ</sub> λ²/c.

</details>

4. When is a log flux axis more useful than a linear one?
<details><summary><b>Answer</b> (try first, then click)</summary>

When the flux covers a big range, so faint regions (like 3–5 µm here) would look flat on a linear axis.

</details>

**Next:** [Notebook 1 · Blackbodies and brightness temperature](https://colab.research.google.com/github/zshakespear/bd-spectra-onramp/blob/main/notebooks/01_blackbodies.ipynb)